# Camada Silver

Este notebook realiza o tratamento e a padronização dos dados armazenados na camada Bronze do MVP de Engenharia de Dados.

As transformações incluem padronização de nomes e tipos, seleção de campos relevantes, organização da granularidade temporal e validação da qualidade dos dados.

Os dados são persistidos em tabelas Delta no schema `silver`, preservando a rastreabilidade das fontes e preparando as informações para a camada Gold.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql import types as T
from pyspark.sql.window import Window

## Configuração do ambiente

O notebook utiliza o catálogo ativo do Databricks e cria o schema Silver, caso ainda não exista. As tabelas Bronze são lidas diretamente do armazenamento persistente, sem executar novamente os processos de coleta.

In [0]:
catalogo = spark.sql(
    "SELECT current_catalog() AS catalogo"
).collect()[0]["catalogo"]

def nome_tabela(schema, tabela):
    return f"`{catalogo}`.`{schema}`.`{tabela}`"

spark.sql(f"""
CREATE SCHEMA IF NOT EXISTS `{catalogo}`.`silver`
COMMENT 'Camada Silver do MVP: dados tratados e padronizados'
""")

print("Catálogo:", catalogo)
print("Schema Silver preparado.")

In [0]:
bronze_criptos = spark.table(nome_tabela("bronze", "criptomoedas"))
bronze_dolar = spark.table(nome_tabela("bronze", "dolar"))
bronze_selic = spark.table(nome_tabela("bronze", "selic"))
bronze_ipca = spark.table(nome_tabela("bronze", "ipca"))
bronze_ibov = spark.table(nome_tabela("bronze", "ibovespa"))

print("Tabelas Bronze carregadas.")

## Padronização das criptomoedas

Os dados de Bitcoin, Ethereum, Solana e XRP são padronizados em uma estrutura diária por ativo.

São mantidos os preços de abertura, máxima, mínima e fechamento, os volumes negociados e a quantidade de negociações. A moeda de cotação é identificada explicitamente como USDT.

O preço de fechamento será utilizado posteriormente para o cálculo de retornos, enquanto os demais campos permitem análises complementares de comportamento de mercado.

In [0]:
silver_criptos = (
    bronze_criptos
    .select(
        F.to_date("data").alias("data"),
        F.col("ativo").cast("string").alias("ativo"),
        F.lit("USDT").alias("moeda_cotacao"),
        F.col("open").cast("double").alias("abertura"),
        F.col("high").cast("double").alias("maxima"),
        F.col("low").cast("double").alias("minima"),
        F.col("close").cast("double").alias("fechamento"),
        F.col("volume").cast("double").alias("volume_base"),
        F.col("quote_asset_volume").cast("double").alias("volume_cotacao"),
        F.col("numero_negociacoes").cast("long").alias("numero_negociacoes"),
        F.col("fonte").cast("string").alias("fonte"),
        F.col("data_ingestao").cast("timestamp").alias("data_ingestao")
    )
)

## Padronização do Dólar/Real

A cotação PTAX de venda é mantida como principal referência cambial do projeto. A cotação de compra também é preservada para rastreabilidade e possíveis análises complementares.

A granularidade corresponde a um registro por dia com cotação disponível.

In [0]:
silver_dolar = (
    bronze_dolar
    .select(
        F.to_date("data").alias("data"),
        F.lit("USD_BRL").alias("indicador"),
        F.col("cotacaoCompra").cast("double").alias("cotacao_compra"),
        F.col("cotacaoVenda").cast("double").alias("cotacao_venda"),
        F.col("fonte").cast("string").alias("fonte"),
        F.col("data_ingestao").cast("timestamp").alias("data_ingestao")
    )
)

## Padronização da Meta Selic

A série SGS 432 representa a meta para a taxa Selic definida pelo Copom, expressa em percentual ao ano.

Os valores são preservados na unidade original, sem conversão para taxa diária nesta etapa.

In [0]:
silver_selic = (
    bronze_selic
    .select(
        F.to_date("data").alias("data"),
        F.lit("SELIC_META").alias("indicador"),
        F.col("selic_meta_pct_aa").cast("double").alias("selic_meta_pct_aa"),
        F.col("fonte").cast("string").alias("fonte"),
        F.col("data_ingestao").cast("timestamp").alias("data_ingestao")
    )
)

## Padronização do IPCA

O IPCA é mantido em sua periodicidade mensal, com valores expressos em percentual de variação no mês.

A coluna `mes_referencia` identifica o mês ao qual o índice se refere. A data de referência não representa necessariamente a data de divulgação do indicador.

A conversão para fator de inflação e o cálculo de retorno real serão realizados posteriormente na camada Gold.

In [0]:
silver_ipca = (
    bronze_ipca
    .select(
        F.trunc(F.to_date("data"), "month").alias("mes_referencia"),
        F.lit("IPCA").alias("indicador"),
        F.col("ipca_pct_mes").cast("double").alias("ipca_pct_mes"),
        F.col("fonte").cast("string").alias("fonte"),
        F.col("data_ingestao").cast("timestamp").alias("data_ingestao")
    )
)

## Padronização do Ibovespa

Os dados oficiais do Ibovespa são organizados por data de pregão, preservando abertura, máxima, mínima, fechamento, valor do índice e oscilação informada pela B3.

O fechamento será utilizado como referência para os cálculos de retorno. Não são criados registros artificiais para fins de semana ou dias sem pregão.

In [0]:
silver_ibov = (
    bronze_ibov
    .select(
        F.to_date("data").alias("data"),
        F.col("ativo").cast("string").alias("ativo"),
        F.col("abertura").cast("double").alias("abertura"),
        F.col("minima").cast("double").alias("minima"),
        F.col("maxima").cast("double").alias("maxima"),
        F.col("fechamento").cast("double").alias("fechamento"),
        F.col("valor_indice").cast("double").alias("valor_indice"),
        F.col("oscilacao").cast("double").alias("oscilacao_fonte"),
        F.col("fonte").cast("string").alias("fonte"),
        F.col("data_ingestao").cast("timestamp").alias("data_ingestao")
    )
)

## Validação da qualidade dos dados

Antes da persistência, são aplicadas verificações de integridade, incluindo:

- ausência de valores nulos em campos obrigatórios;
- inexistência de duplicidades na chave de cada tabela;
- validade dos valores numéricos;
- consistência dos preços de mercado;
- cobertura temporal dentro do período definido.

As validações interrompem a execução caso seja encontrada alguma inconsistência. Dessa forma, dados inválidos não são gravados silenciosamente na Silver.

In [0]:
inicio = "2021-01-01"
fim = "2025-12-31"

def validar_tabela(df, nome, chave, obrigatorias, regras=None):
    print(f"\n=== VALIDAÇÃO: {nome} ===")

    quantidade = df.count()

    nulos = {
        coluna: df.filter(F.col(coluna).isNull()).count()
        for coluna in obrigatorias
    }

    duplicados = (
        df.groupBy(*chave)
        .count()
        .filter(F.col("count") > 1)
        .count()
    )

    print("Registros:", quantidade)
    print("Nulos:", nulos)
    print("Chaves duplicadas:", duplicados)

    if quantidade == 0:
        raise ValueError(f"{nome}: tabela vazia.")

    if any(valor > 0 for valor in nulos.values()):
        raise ValueError(f"{nome}: existem valores nulos obrigatórios.")

    if duplicados > 0:
        raise ValueError(f"{nome}: existem chaves duplicadas.")

    for descricao, condicao in (regras or []):
        invalidos = df.filter(~condicao).count()
        print(f"{descricao}: {invalidos} inválidos")

        if invalidos > 0:
            raise ValueError(f"{nome}: falha na regra {descricao}.")

    print("Qualidade validada.")

In [0]:
validar_tabela(
    silver_criptos,
    "Criptomoedas",
    ["data", "ativo"],
    [
        "data", "ativo", "moeda_cotacao",
        "abertura", "maxima", "minima", "fechamento",
        "volume_base", "volume_cotacao", "numero_negociacoes"
    ],
    [
        (
            "Período",
            F.col("data").between(inicio, fim)
        ),
        (
            "Preços positivos",
            (F.col("abertura") > 0) &
            (F.col("maxima") > 0) &
            (F.col("minima") > 0) &
            (F.col("fechamento") > 0)
        ),
        (
            "Consistência OHLC",
            (F.col("maxima") >= F.greatest("abertura", "fechamento", "minima")) &
            (F.col("minima") <= F.least("abertura", "fechamento", "maxima"))
        ),
        (
            "Volumes e negociações não negativos",
            (F.col("volume_base") >= 0) &
            (F.col("volume_cotacao") >= 0) &
            (F.col("numero_negociacoes") >= 0)
        )
    ]
)

In [0]:
validar_tabela(
    silver_dolar,
    "Dólar/Real",
    ["data"],
    ["data", "cotacao_compra", "cotacao_venda"],
    [
        ("Período", F.col("data").between(inicio, fim)),
        (
            "Cotações positivas",
            (F.col("cotacao_compra") > 0) &
            (F.col("cotacao_venda") > 0)
        ),
        (
            "Compra menor ou igual à venda",
            F.col("cotacao_compra") <= F.col("cotacao_venda")
        )
    ]
)

validar_tabela(
    silver_selic,
    "Meta Selic",
    ["data"],
    ["data", "selic_meta_pct_aa"],
    [
        ("Período", F.col("data").between(inicio, fim)),
        (
            "Taxa não negativa",
            F.col("selic_meta_pct_aa") >= 0
        )
    ]
)

validar_tabela(
    silver_ipca,
    "IPCA",
    ["mes_referencia"],
    ["mes_referencia", "ipca_pct_mes"],
    [
        (
            "Período",
            F.col("mes_referencia").between(inicio, fim)
        ),
        (
            "Mês de referência",
            F.dayofmonth("mes_referencia") == 1
        ),
        (
            "Fator de inflação positivo",
            F.col("ipca_pct_mes") > -100
        )
    ]
)

In [0]:
validar_tabela(
    silver_ibov,
    "Ibovespa",
    ["data"],
    [
        "data", "ativo", "abertura", "minima",
        "maxima", "fechamento", "valor_indice"
    ],
    [
        ("Período", F.col("data").between(inicio, fim)),
        (
            "Preços positivos",
            (F.col("abertura") > 0) &
            (F.col("minima") > 0) &
            (F.col("maxima") > 0) &
            (F.col("fechamento") > 0)
        ),
        (
            "Consistência OHLC",
            (F.col("maxima") >= F.greatest("abertura", "fechamento", "minima")) &
            (F.col("minima") <= F.least("abertura", "fechamento", "maxima"))
        )
    ]
)

## Persistência das tabelas Silver

Após a validação, os DataFrames tratados são gravados como tabelas Delta gerenciadas no schema Silver.

O modo `overwrite` é utilizado para permitir a reprodução integral do notebook, substituindo as tabelas anteriormente geradas. As tabelas Bronze permanecem inalteradas.

In [0]:
tabelas_silver = {
    "criptomoedas": silver_criptos,
    "dolar": silver_dolar,
    "selic": silver_selic,
    "ipca": silver_ipca,
    "ibovespa": silver_ibov
}

for nome, df in tabelas_silver.items():
    (
        df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(nome_tabela("silver", nome))
    )

    print(f"Tabela gravada: silver.{nome}")

## Documentação das tabelas

As tabelas Silver são documentadas no catálogo de dados com descrições de finalidade, origem e granularidade.

Essa documentação facilita a compreensão do modelo, a rastreabilidade das fontes e a utilização das tabelas nas etapas analíticas.

In [0]:
comentarios_tabelas = {
    "criptomoedas": (
        "Dados diários tratados de BTC, ETH, SOL e XRP, "
        "cotados em USDT. Granularidade: um ativo por dia. "
        "Fonte: Binance API."
    ),
    "dolar": (
        "Cotações PTAX de compra e venda do dólar. "
        "Granularidade: um registro por dia com cotação. "
        "Fonte: Banco Central do Brasil."
    ),
    "selic": (
        "Meta Selic definida pelo Copom, em percentual ao ano. "
        "Granularidade diária. Fonte: BCB, SGS 432."
    ),
    "ipca": (
        "Variação mensal do IPCA em percentual. "
        "Granularidade: um registro por mês de referência. "
        "Fonte: BCB, SGS 433."
    ),
    "ibovespa": (
        "Dados de mercado do Ibovespa por pregão, incluindo OHLC "
        "e valor do índice. Fonte: B3 IndexReport."
    )
}

for tabela, comentario in comentarios_tabelas.items():
    spark.sql(
        f"COMMENT ON TABLE {nome_tabela('silver', tabela)} "
        f"IS '{comentario}'"
    )

print("Descrições das tabelas registradas no catálogo.")

## Validação da persistência

As tabelas Silver são consultadas novamente após a gravação para confirmar a existência dos dados, as quantidades de registros e os períodos disponíveis.

A validação compara os resultados persistidos com as quantidades obtidas na camada Bronze, garantindo que o tratamento não tenha eliminado registros indevidamente.

In [0]:
print("=== VALIDAÇÃO FINAL DA CAMADA SILVER ===")

for nome, df_origem in tabelas_silver.items():
    tabela = spark.table(nome_tabela("silver", nome))

    quantidade_silver = tabela.count()
    quantidade_origem = df_origem.count()

    print(
        f"{nome}: {quantidade_silver} registros "
        f"| origem tratada: {quantidade_origem}"
    )

    if quantidade_silver != quantidade_origem:
        raise ValueError(
            f"Divergência na persistência da tabela {nome}."
        )

print("\nTodas as tabelas Silver foram persistidas com sucesso.")

## Resultado da camada Silver

As cinco fontes foram tratadas, padronizadas e persistidas em tabelas Delta no schema Silver.

Foram aplicadas verificações de integridade, unicidade, valores obrigatórios, consistência numérica e cobertura temporal. As frequências originais foram preservadas, evitando preenchimentos artificiais e integração inadequada de dados diários e mensais.

As tabelas Silver estão preparadas para a construção da camada Gold, na qual serão calculados indicadores de retorno, volatilidade, retorno real e correlação, além da integração entre ativos financeiros e variáveis macroeconômicas.